# Final_03_user.ipynb 步驟目錄
---
| Step | 內容 | 使用資料 | 輸出檔案 |
|---:|---|---|---|
| 1 | 讀取 commander 所需資料與路網 | `taipei_drive.graphml`, `road_grid_traveling.csv`, `taipei_road_grid_segments.geojson`, `taipei_shelters_join_500m.geojson` | - |
| 2 | 建立災害情境 routing graph | `G`, `road_traveling`, `road_grid_segments` | `G_route` |
| 3 | 設定使用者位置並搜尋候選避難所 | 使用者經緯度、`shelters`, `G_route` | - |
| 4 | 計算至避難所的最短路徑 | `G_route`, `commander_travel_time`, `candidate_shelters` | `route_gdf` |
| 5 | 選出建議避難所前三名 | `route_result_df`, `select_by` | `nearest_3_shelters` |
| 6 | 計算正常與災害情境 5 分鐘可達範圍 | `G`, `G_route`, 使用者位置 | `normal_iso_poly`, `disaster_iso_poly` |
| 7 | 建立避難所、路線與可達範圍地圖 | `shelters`, `route_gdf`, `taipei_boundary`, `normal_iso_poly`, `disaster_iso_poly` | `output/user_accessibility_routes_map.html` |

### Commander 資料讀取與路網準備
---
本段程式讀取第三階段指揮官模式所需的資料，並建立災害情境下可用於路徑規劃的道路 graph。

| 變數名稱 | 來源資料 | 說明 |
|---|---|---|
| `G` | `output/taipei_drive.graphml` | 臺北市原始道路 graph，並轉換為 `EPSG:3826` |
| `road_traveling` | `output/road_grid_traveling.csv` | 每段 road-grid 的災後車速、通行時間與道路狀態 |
| `road_grid_segments` | `output/taipei_road_grid_segments.geojson` | 被 grid 切分後的道路 geometry，保留 `road_grid_id` 與原始 edge ID |
| `roads` | `road_grid_segments` + `road_traveling` | 合併道路 geometry 與災後通行屬性的完整道路資料 |
| `edge_cost` | `roads` 彙整結果 | 將 road-grid 分段通行時間彙整回原始 graph edge |
| `G_route` | `G` 移除封閉 / 缺資料道路後 | 災害情境下實際用於路徑規劃的道路 graph |
| `shelters` | `output/taipei_shelters_join_500m.geojson` | 臺北市避難所資料，並已連結最近道路 |
| `taipei_boundary` | `data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp` | 臺北市各行政區邊界 |
| `boundary` | `taipei_boundary.dissolve()` | 臺北市整體行政邊界 |

其中，`commander_travel_time` 是本階段最重要的路徑成本欄位，代表災害情境下每條道路 edge 的通行時間。程式會將封閉道路或缺少通行時間的道路移除，產生 `G_route`，供後續避難所路線規劃使用。

In [15]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx

TARGET_CRS = "EPSG:3826"

# ===== 路徑設定 =====
graph_path = Path("output/taipei_drive.graphml")
road_traveling_path = Path("output/road_grid_traveling.csv")
road_grid_path = Path("output/taipei_road_grid_segments.geojson")
shelter_path = Path("output/taipei_shelters_join_500m.geojson")
town_path = Path("data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp")

# ===== 檢查檔案是否存在 =====
required_paths = {
    "road_graph": graph_path,
    "road_grid_traveling": road_traveling_path,
    "road_grid_segments": road_grid_path,
    "shelters": shelter_path,
    "town": town_path,
}

missing_paths = {
    name: path
    for name, path in required_paths.items()
    if not path.exists()
}

if missing_paths:
    raise FileNotFoundError(f"找不到檔案: {missing_paths}")

# ===== 1) 讀取道路 graph =====
G = ox.load_graphml(graph_path)
G = ox.project_graph(G, to_crs=TARGET_CRS)

# ===== 2) 讀取 road-grid 通行屬性表 =====
road_traveling = pd.read_csv(road_traveling_path)

# ===== 3) 讀取 road-grid geometry，並把通行屬性接回來 =====
road_grid_segments = gpd.read_file(road_grid_path).to_crs(TARGET_CRS)

roads = road_grid_segments.merge(
    road_traveling,
    on="road_grid_id",
    how="left",
    suffixes=("", "_travel"),
)

roads = gpd.GeoDataFrame(
    roads,
    geometry="geometry",
    crs=TARGET_CRS,
)

# ===== 4) 將 road-grid 通行時間彙整回原始 graph edge =====
def clean_edge_id(x):
    if pd.isna(x):
        return None
    try:
        return str(int(float(x)))
    except Exception:
        return str(x)

roads["u_key"] = roads["u"].apply(clean_edge_id)
roads["v_key"] = roads["v"].apply(clean_edge_id)
roads["key_key"] = roads["key"].apply(clean_edge_id)

edge_cost = (
    roads
    .groupby(["u_key", "v_key", "key_key"], as_index=False)
    .agg(
        commander_travel_time=("post_travel_time", "sum"),
        post_speed_kph_min=("post_speed_kph", "min"),
        closed_count=("road_status", lambda s: (s == "closed").sum()),
        segment_count=("road_grid_id", "count"),
    )
)

edge_cost["is_closed"] = (
    (edge_cost["closed_count"] > 0)
    | (~np.isfinite(edge_cost["commander_travel_time"]))
)

edge_cost_lookup = {
    (row.u_key, row.v_key, row.key_key): row
    for row in edge_cost.itertuples(index=False)
}

# ===== 5) 寫入 graph edge 權重 =====
matched_commander_edges = 0

for u, v, key, data in G.edges(keys=True, data=True):
    lookup_key = (
        clean_edge_id(u),
        clean_edge_id(v),
        clean_edge_id(key),
    )

    row = edge_cost_lookup.get(lookup_key)

    if row is None:
        data["commander_travel_time"] = np.inf
        data["commander_status"] = "missing"
    elif bool(row.is_closed):
        data["commander_travel_time"] = np.inf
        data["commander_status"] = "closed"
    else:
        data["commander_travel_time"] = float(row.commander_travel_time)
        data["commander_status"] = "open"
        matched_commander_edges += 1

# ===== 6) 建立 routing 用 graph：移除封閉或缺資料道路 =====
G_route = G.copy()

closed_edges = [
    (u, v, key)
    for u, v, key, data in G_route.edges(keys=True, data=True)
    if not np.isfinite(data.get("commander_travel_time", np.inf))
]

G_route.remove_edges_from(closed_edges)

print("Matched commander graph edges:", matched_commander_edges)

# ===== 7) 讀取避難所 =====
shelters = gpd.read_file(shelter_path).to_crs(TARGET_CRS)

# ===== 8) 讀取臺北市行政邊界 =====
town = gpd.read_file(town_path)
town.columns = [str(c).strip().strip("'").strip('"') for c in town.columns]

if "COUNTYCODE" in town.columns:
    taipei_boundary = town[
        town["COUNTYCODE"].astype(str).str.contains("63000", na=False)
    ].copy()
elif "COUNTYNAME" in town.columns:
    taipei_boundary = town[
        town["COUNTYNAME"].astype(str).str.contains("臺北市|台北市", na=False, regex=True)
    ].copy()
else:
    taipei_boundary = town.copy()

taipei_boundary = taipei_boundary.to_crs(TARGET_CRS)
boundary = taipei_boundary.dissolve()

# ===== 9) 基本檢查 =====
print("Road graph nodes:", len(G.nodes))
print("Road graph edges:", len(G.edges))
print("Routing graph edges:", len(G_route.edges))
print("Removed closed/missing edges:", len(closed_edges))
print("Road traveling rows:", len(road_traveling))
print("Road geometry rows:", len(road_grid_segments))
print("Merged roads rows:", len(roads))
print("Shelters:", len(shelters))
print("Taipei boundary:", len(taipei_boundary))

Matched commander graph edges: 74509
Road graph nodes: 31582
Road graph edges: 74509
Routing graph edges: 74509
Removed closed/missing edges: 0
Road traveling rows: 101018
Road geometry rows: 101018
Merged roads rows: 101018
Shelters: 278
Taipei boundary: 12


### 最近避難所與路線分析
---
本段程式以使用者輸入的經緯度作為出發點，先將位置轉換到臺灣常用座標系統 `EPSG:3826`，再尋找路網中距離使用者最近的節點作為起點。

接著，程式會將每個避難所對應到最近的道路節點，並先用直線距離篩選出距離使用者最近的 30 個候選避難所，以降低路徑計算時間。

在候選避難所中，系統使用災害後道路通行時間 `commander_travel_time` 作為路徑成本，透過最短路徑演算法計算從使用者位置到各避難所的實際行車路線與旅行時間。

最後依照設定的 `select_by` 條件排序：

| `select_by` | 說明 |
|---|---|
| `straight_distance` | 選擇直線距離最近的三個避難所 |
| `travel_time` | 選擇災害情境下行車時間最短的三個避難所 |

輸出結果包含三個建議避難所的名稱、地址、行政區、收容人數、直線距離、路線長度與預估行車時間，並建立 `route_gdf` 作為後續地圖視覺化使用。

In [16]:
import osmnx as ox
import networkx as nx
from shapely.geometry import LineString
from shapely.ops import linemerge

# ===== 輸入使用者位置 WGS84 =====
user_lon = 121.533355
user_lat = 25.016903
# 可選：
# "straight_distance" = 直線距離最近三個
# "travel_time" = 災害情境下行車時間最短三個
select_by = "travel_time"

user_gdf = gpd.GeoDataFrame(
    {"id": ["user"]},
    geometry=gpd.points_from_xy([user_lon], [user_lat]),
    crs="EPSG:4326",
).to_crs(TARGET_CRS)

user_point = user_gdf.geometry.iloc[0]

# 使用災害後 routing graph
origin_node = ox.nearest_nodes(G_route, X=user_point.x, Y=user_point.y)

# 每個 shelter 找最近 graph node
shelters_route = shelters.copy()
shelters_route["shelter_node"] = shelters_route.geometry.apply(
    lambda geom: ox.nearest_nodes(G_route, X=geom.x, Y=geom.y)
)

# 先抓直線距離最近 30 個，避免全部跑 shortest path 太慢
shelters_route["straight_dist_m"] = shelters_route.geometry.distance(user_point)
candidate_shelters = shelters_route.sort_values("straight_dist_m").head(30).copy()


def get_edge_geometry(G_input, u, v, weight_col="commander_travel_time"):
    edge_data = G_input.get_edge_data(u, v)

    if edge_data is None:
        return None, np.inf, np.inf

    best_key = min(
        edge_data,
        key=lambda k: edge_data[k].get(weight_col, np.inf),
    )

    data = edge_data[best_key]

    if "geometry" in data and data["geometry"] is not None:
        geom = data["geometry"]
    else:
        ux = G_input.nodes[u]["x"]
        uy = G_input.nodes[u]["y"]
        vx = G_input.nodes[v]["x"]
        vy = G_input.nodes[v]["y"]
        geom = LineString([(ux, uy), (vx, vy)])

    travel_time = data.get(weight_col, np.inf)
    length_m = data.get("length", geom.length)

    return geom, travel_time, length_m


def route_to_geometry(G_input, route, weight_col="commander_travel_time"):
    route_parts = []
    total_time_sec = 0
    total_length_m = 0

    for u, v in zip(route[:-1], route[1:]):
        geom, travel_time, length_m = get_edge_geometry(
            G_input,
            u,
            v,
            weight_col=weight_col,
        )

        if geom is None or not np.isfinite(travel_time):
            continue

        route_parts.append(geom)
        total_time_sec += travel_time
        total_length_m += length_m

    if len(route_parts) == 0:
        return None, np.inf, np.inf

    return linemerge(route_parts), total_time_sec, total_length_m


route_results = []

for _, row in candidate_shelters.iterrows():
    shelter_node = row["shelter_node"]

    try:
        route = nx.shortest_path(
            G_route,
            source=origin_node,
            target=shelter_node,
            weight="commander_travel_time",
        )

        route_geom, travel_time_sec, route_length_m = route_to_geometry(
            G_route,
            route,
            weight_col="commander_travel_time",
        )

        if route_geom is None or not np.isfinite(travel_time_sec):
            continue

        route_results.append({
            "shelter_id": row.get("shelter_id"),
            "shelter_name": row.get("避難收容處所名稱"),
            "address": row.get("避難收容處所地址"),
            "capacity": row.get("預計收容人數"),
            "town": row.get("TOWNNAME"),
            "straight_dist_m": row["straight_dist_m"],
            "route_length_m": route_length_m,
            "travel_time_min": travel_time_sec / 60,
            "route": route,
            "geometry": route_geom,
        })

    except (nx.NetworkXNoPath, nx.NodeNotFound):
        continue


route_result_df = pd.DataFrame(route_results)

if len(route_result_df) == 0:
    raise ValueError("找不到可通行到避難所的路線")

if select_by == "straight_distance":
    nearest_3_shelters = (
        route_result_df
        .sort_values("straight_dist_m")
        .head(3)
        .reset_index(drop=True)
    )
elif select_by == "travel_time":
    nearest_3_shelters = (
        route_result_df
        .sort_values("travel_time_min")
        .head(3)
        .reset_index(drop=True)
    )
else:
    raise ValueError("select_by 只能是 'straight_distance' 或 'travel_time'")

nearest_3_shelters["route_rank"] = nearest_3_shelters.index + 1

route_gdf = gpd.GeoDataFrame(
    nearest_3_shelters.drop(columns=["route"]),
    geometry="geometry",
    crs=TARGET_CRS,
)

display(
    nearest_3_shelters[
        [
            "route_rank",
            "shelter_id",
            "shelter_name",
            "town",
            "address",
            "capacity",
            "straight_dist_m",
            "route_length_m",
            "travel_time_min",
        ]
    ]
)

,route_rank,shelter_id,shelter_name,town,address,capacity,straight_dist_m,route_length_m,travel_time_min
0,1,34,臺北市中正運動中心,中正區,信義路一段1號,250,391.711470,466.989273,0.473035
1,2,40,古亭國小,大安區,大安區羅斯福路3段,62,645.749674,921.629913,0.933066
2,3,32,臺北市大安運動中心,大安區,辛亥路三段55號,389,837.303112,1095.371964,1.191549


### 五分鐘車程可達範圍與避難所可及性分析
---
本 cell 主要用來比較災前正常情境與災後道路受災情境下，使用者位置周邊五分鐘車程內的可達範圍與避難所可及性。

首先，程式會根據道路的正常行車速率建立 `G_normal` routing graph，並以災後通行時間 `commander_travel_time` 建立災後 routing graph。接著以使用者所在節點作為起點，分別計算正常情境與災後情境下，五分鐘內可到達的道路節點。

在空間分析部分，程式會將五分鐘內可到達的道路節點與道路邊線轉換為 isochrone polygon，作為五分鐘車程可達範圍。並進一步計算災前與災後的五分鐘車程面積、面積縮小量，以及相較災前的縮小比例。

此外，程式也會判斷各避難所是否位於五分鐘車程範圍內，統計災前與災後五分鐘內可到達的避難所總數，並列出兩種情境下最近的三個避難所及其車程時間。

最後，分析結果會被整理成摘要表格，並透過 Folium 製作互動式地圖。地圖中包含台北市邊界、災前與災後五分鐘車程範圍、使用者位置、避難所位置，以及最近三個避難所的路徑，方便比較災害前後道路可及性與避難可達性的變化。

---
### `user_accessbility_routes_map.html` 圖層說明
---

本地圖主要包含三個分析圖層，用來比較正常情境與災害情境下的避難可及性。

| 圖層 | 說明 |
|---|---|
| Taipei Boundary | 臺北市行政邊界，用來標示本次分析範圍 |
| Normal 5-min drive area | 正常情境下，從使用者所在位置出發，5 分鐘車程內可到達的範圍 |
| Disaster 5-min drive area | 災害情境下，考量道路降速、淹水與坡地風險後，從使用者所在位置出發，5 分鐘車程內可到達的範圍 |

底圖使用 `cartodbpositron`，作為道路與地理位置的背景參考。

In [17]:
import folium

# ===== 0) 參數設定 =====
cutoff_min = 5
cutoff_sec = cutoff_min * 60

# ===== 1) 建立 normal routing graph =====
roads = roads.copy()

roads["normal_access_travel_time"] = np.where(
    roads["normal_speed_kph"] > 0,
    roads["segment_length_m"] / (roads["normal_speed_kph"] * 1000 / 3600),
    np.inf,
)

if "u_key" not in roads.columns:
    roads["u_key"] = roads["u"].apply(clean_edge_id)
    roads["v_key"] = roads["v"].apply(clean_edge_id)
    roads["key_key"] = roads["key"].apply(clean_edge_id)

normal_edge_cost = (
    roads
    .groupby(["u_key", "v_key", "key_key"], as_index=False)
    .agg(normal_access_travel_time=("normal_access_travel_time", "sum"))
)

normal_lookup = {
    (row.u_key, row.v_key, row.key_key): float(row.normal_access_travel_time)
    for row in normal_edge_cost.itertuples(index=False)
}

G_normal = G.copy()

for u, v, key, data in G_normal.edges(keys=True, data=True):
    lookup_key = (
        clean_edge_id(u),
        clean_edge_id(v),
        clean_edge_id(key),
    )
    data["normal_access_travel_time"] = normal_lookup.get(lookup_key, np.inf)

normal_remove_edges = [
    (u, v, key)
    for u, v, key, data in G_normal.edges(keys=True, data=True)
    if not np.isfinite(data.get("normal_access_travel_time", np.inf))
]

G_normal.remove_edges_from(normal_remove_edges)

# ===== 2) 計算 normal / disaster 5 分鐘內可達節點 =====
normal_lengths = nx.single_source_dijkstra_path_length(
    G_normal,
    origin_node,
    cutoff=cutoff_sec,
    weight="normal_access_travel_time",
)

disaster_lengths = nx.single_source_dijkstra_path_length(
    G_route,
    origin_node,
    cutoff=cutoff_sec,
    weight="commander_travel_time",
)

print("Normal 5-min reachable nodes:", len(normal_lengths))
print("Disaster 5-min reachable nodes:", len(disaster_lengths))

# ===== 3) 將可達節點轉成 isochrone polygon =====
def make_isochrone_polygon(G_input, reachable_lengths, node_buffer_m=120, edge_buffer_m=50):
    reachable_nodes = set(reachable_lengths.keys())

    if len(reachable_nodes) == 0:
        return None

    nodes_gdf = ox.graph_to_gdfs(G_input, nodes=True, edges=False)

    geoms = []

    node_buffers = (
        nodes_gdf
        .loc[nodes_gdf.index.isin(reachable_nodes), "geometry"]
        .buffer(node_buffer_m)
    )
    geoms.extend(list(node_buffers))

    if len(G_input.edges) > 0:
        edges_gdf = ox.graph_to_gdfs(G_input, nodes=False, edges=True)

        edge_u = edges_gdf.index.get_level_values(0)
        edge_v = edges_gdf.index.get_level_values(1)

        edge_buffers = (
            edges_gdf
            .loc[
                edge_u.isin(reachable_nodes) & edge_v.isin(reachable_nodes),
                "geometry",
            ]
            .buffer(edge_buffer_m)
        )
        geoms.extend(list(edge_buffers))

    if len(geoms) == 0:
        return None

    return gpd.GeoSeries(geoms, crs=TARGET_CRS).union_all()


normal_iso_poly = make_isochrone_polygon(G_normal, normal_lengths)
disaster_iso_poly = make_isochrone_polygon(G_route, disaster_lengths)

normal_iso = gpd.GeoDataFrame(
    {"scenario": ["normal_5min"]},
    geometry=[normal_iso_poly],
    crs=TARGET_CRS,
)

disaster_iso = gpd.GeoDataFrame(
    {"scenario": ["disaster_5min"]},
    geometry=[disaster_iso_poly],
    crs=TARGET_CRS,
)

# ===== 4) 標註避難所是否在 5 分鐘內可達 =====
shelters_map = shelters.copy()

shelters_map["shelter_node"] = shelters_map.geometry.apply(
    lambda geom: ox.nearest_nodes(G, X=geom.x, Y=geom.y)
)

shelters_map["normal_time_min"] = shelters_map["shelter_node"].map(normal_lengths) / 60
shelters_map["disaster_time_min"] = shelters_map["shelter_node"].map(disaster_lengths) / 60

shelters_map["normal_5min"] = shelters_map["shelter_node"].isin(normal_lengths.keys())
shelters_map["disaster_5min"] = shelters_map["shelter_node"].isin(disaster_lengths.keys())

# ===== 5 分鐘車程統計表 =====

normal_area_m2 = normal_iso.geometry.area.iloc[0]
disaster_area_m2 = disaster_iso.geometry.area.iloc[0]

normal_area_km2 = normal_area_m2 / 1_000_000
disaster_area_km2 = disaster_area_m2 / 1_000_000

area_reduction_km2 = normal_area_km2 - disaster_area_km2
shrink_ratio = (normal_area_m2 - disaster_area_m2) / normal_area_m2

normal_shelter_count = shelters_map["normal_5min"].sum()
disaster_shelter_count = shelters_map["disaster_5min"].sum()

# 避難所名稱欄位，依你的資料欄位名稱調整
shelter_name_col = "避難收容處所名稱"

# 如果欄位名稱是亂碼或不同，改用自動抓第一個含「名稱」的欄位
if shelter_name_col not in shelters_map.columns:
    possible_name_cols = [c for c in shelters_map.columns if "名稱" in str(c) or "name" in str(c).lower()]
    shelter_name_col = possible_name_cols[0] if possible_name_cols else shelters_map.columns[0]

normal_nearest = (
    shelters_map[shelters_map["normal_5min"]]
    .sort_values("normal_time_min")
    .head(3)
)

disaster_nearest = (
    shelters_map[shelters_map["disaster_5min"]]
    .sort_values("disaster_time_min")
    .head(3)
)

normal_nearest_name = (
    normal_nearest[shelter_name_col].iloc[0]
    if len(normal_nearest) > 0 else "N/A"
)

disaster_nearest_name = (
    disaster_nearest[shelter_name_col].iloc[0]
    if len(disaster_nearest) > 0 else "N/A"
)

def format_nearest_shelters(df, time_col):
    if len(df) == 0:
        return "N/A"

    return "；".join(
        f"{row[shelter_name_col]}（{row[time_col]:.1f} 分鐘）"
        for _, row in df.iterrows()
    )

normal_nearest_text = format_nearest_shelters(
    normal_nearest,
    "normal_time_min"
)

disaster_nearest_text = format_nearest_shelters(
    disaster_nearest,
    "disaster_time_min"
)

normal_nearest_time = (
    normal_nearest["normal_time_min"].iloc[0]
    if len(normal_nearest) > 0 else np.nan
)

disaster_nearest_time = (
    disaster_nearest["disaster_time_min"].iloc[0]
    if len(disaster_nearest) > 0 else np.nan
)

access_summary = pd.DataFrame({
    "情境": ["災前 / 正常", "災後"],
    "五分鐘車程面積_km2": [normal_area_km2, disaster_area_km2],
    "相較災前縮小比例": ["0.00%", f"{shrink_ratio:.2%}"],
    "五分鐘車程內避難所總數": [normal_shelter_count, disaster_shelter_count],
    "最易到達的三個避難所": [normal_nearest_text, disaster_nearest_text],
    "最近避難所車程_分鐘": [normal_nearest_time, disaster_nearest_time],
})

display(access_summary)

# ===== 5) 轉 WGS84 給 folium =====
boundary_wgs84 = boundary.to_crs(epsg=4326)
normal_iso_wgs84 = normal_iso.to_crs(epsg=4326)
disaster_iso_wgs84 = disaster_iso.to_crs(epsg=4326)
shelters_wgs84 = shelters_map.to_crs(epsg=4326)
route_gdf_wgs84 = route_gdf.to_crs(epsg=4326)

# ===== 6) 建立 folium 地圖 =====
m = folium.Map(
    location=[user_lat, user_lon],
    zoom_start=14,
    tiles="cartodbpositron",
)

# 臺北市邊界
folium.GeoJson(
    boundary_wgs84[["geometry"]].to_json(),
    name="Taipei Boundary",
    style_function=lambda feature: {
        "color": "#111111",
        "weight": 2,
        "fillOpacity": 0,
    },
).add_to(m)

# Normal 5-min area
folium.GeoJson(
    normal_iso_wgs84.to_json(),
    name="Normal 5-min drive area",
    style_function=lambda feature: {
        "color": "#2166ac",
        "weight": 2,
        "fillColor": "#67a9cf",
        "fillOpacity": 0.25,
    },
).add_to(m)

# Disaster 5-min area
folium.GeoJson(
    disaster_iso_wgs84.to_json(),
    name="Disaster 5-min drive area",
    style_function=lambda feature: {
        "color": "#b2182b",
        "weight": 2,
        "fillColor": "#ef8a62",
        "fillOpacity": 0.35,
    },
).add_to(m)

# 使用者位置
folium.Marker(
    location=[user_lat, user_lon],
    popup="User location",
    tooltip="User location",
    icon=folium.Icon(color="red", icon="user"),
).add_to(m)

# ===== 7) 避難所圖層 =====
def shelter_color(row):
    if row["disaster_5min"]:
        return "#1a9850"   # 災害後 5 分鐘內可達
    if row["normal_5min"]:
        return "#fdae61"   # normal 可達，但災害後不可達
    return "#999999"       # 5 分鐘內不可達


shelter_fg = folium.FeatureGroup(name="Shelters", show=True)

for _, row in shelters_wgs84.iterrows():
    name = row.get("避難收容處所名稱", "N/A")
    addr = row.get("避難收容處所地址", "N/A")
    cap = row.get("預計收容人數", "N/A")
    town = row.get("TOWNNAME", "N/A")

    normal_time = row.get("normal_time_min")
    disaster_time = row.get("disaster_time_min")

    normal_text = "N/A" if pd.isna(normal_time) else f"{normal_time:.1f} min"
    disaster_text = "N/A" if pd.isna(disaster_time) else f"{disaster_time:.1f} min"

    popup_html = f"""
    <b>{name}</b><br>
    行政區: {town}<br>
    地址: {addr}<br>
    預計收容人數: {cap}<br>
    Normal travel time: {normal_text}<br>
    Disaster travel time: {disaster_text}
    """

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="#111111",
        weight=1,
        fill=True,
        fill_color=shelter_color(row),
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=360),
        tooltip=name,
    ).add_to(shelter_fg)

shelter_fg.add_to(m)

# ===== 8) 最近三個避難所路線圖層 =====
route_colors = ["#d7191c", "#2c7bb6", "#fdae61"]

route_fg = folium.FeatureGroup(
    name="Nearest 3 Shelter Routes",
    show=True,
)

for i, row in route_gdf_wgs84.iterrows():
    color = route_colors[i % len(route_colors)]

    popup_html = f"""
    <b>Route rank:</b> {row["route_rank"]}<br>
    <b>Shelter:</b> {row["shelter_name"]}<br>
    <b>Town:</b> {row["town"]}<br>
    <b>Travel time:</b> {row["travel_time_min"]:.1f} min<br>
    <b>Route length:</b> {row["route_length_m"]:.0f} m<br>
    <b>Straight distance:</b> {row["straight_dist_m"]:.0f} m
    """

    folium.GeoJson(
        row.geometry.__geo_interface__,
        name=f"Route {row['route_rank']}",
        style_function=lambda feature, color=color: {
            "color": color,
            "weight": 5,
            "opacity": 0.9,
        },
        popup=folium.Popup(popup_html, max_width=360),
        tooltip=f"Route {row['route_rank']} - {row['travel_time_min']:.1f} min",
    ).add_to(route_fg)

route_fg.add_to(m)

# ===== 9) 輸出 =====
folium.LayerControl(collapsed=False).add_to(m)

map_path = "output/user_accessibility_routes_map.html"
m.save(map_path)

print("Saved:", map_path)

m

Normal 5-min reachable nodes: 10126
Disaster 5-min reachable nodes: 9509


,情境,五分鐘車程面積_km2,相較災前縮小比例,五分鐘車程內避難所總數,最易到達的三個避難所,最近避難所車程_分鐘
0,災前 / 正常,37.521114,0.00%,108,臺北市中正運動中心（0.5 分鐘）；古亭國小（0.9 分鐘）；民族國中（1.2 分鐘）,0.473035
1,災後,34.774043,7.32%,101,臺北市中正運動中心（0.5 分鐘）；古亭國小（0.9 分鐘）；臺北市大安運動中心（1.2 分鐘）,0.473035


Saved: output/user_accessibility_routes_map.html


## 結案報告

本專案以臺北市為研究範圍，建立一套災害情境下的道路可及性與避難支援分析流程。前處理階段先建立 250m × 250m 網格，整合地形高程、坡度、河川分布、歷史淹水紀錄與避難所資料，計算各網格的淹水風險與邊坡風險。災害階段則讀取指定時間雨量資料，使用 Kriging 將 Past 3hr、Past 6hr、Past 24hr 雨量推估到每個網格，分別代表即時行車影響、淹水壓力與坡地災害壓力。

道路分析部分將臺北市路網切分到網格內，並依道路類型、雨量、淹水風險與邊坡風險計算災後車速與通行時間，輸出 `road_grid_traveling.csv` 作為後續路徑規劃依據。最後在指揮官模式中，使用者可輸入所在經緯度，系統會分析正常與災害情境下 5 分鐘車程可及範圍，並找出可通行時間最短的三個避難所，同時輸出互動式地圖，顯示避難所、路線與可及範圍差異。

身為避難者，應先確認自身位置，查看最近且災害情境下仍可通行的避難所，不應只依直線距離判斷，而要依實際行車時間與道路狀態選擇避難路線。若最近避難所因道路受阻而不可達，應改採系統建議的第二或第三順位避難所。

身為指揮官，應優先關注災害後可及範圍縮小的區域、通往避難所的低速或封閉路段，以及高淹水風險與高坡地風險網格。決策上可依據道路通行時間、避難所分布與風險熱區，安排交通管制、救援路線、避難引導與復原優先順序。